### Generate & scraping json data

Automating the process of scraping data from Wikipedia and generating data in JSON files because I'm lazy.


#### Imports


In [47]:
import uuid
import json
import requests
import pandas as pd

from io import StringIO
from bs4 import BeautifulSoup
from dotenv import dotenv_values

#### Look-ups


In [95]:
with open("../data/groups.json", "r") as f:
    groups_data = json.load(f)
    f.close()

with open("../data/teams.json", "r") as f:
    teams_data = json.load(f)
    f.close()

with open("../data/stages.json", "r") as f:
    stages_data = json.load(f)
    f.close()

In [96]:
groups_lookup = pd.DataFrame(groups_data["groups"])
teams_lookup = pd.DataFrame(teams_data["teams"])
stages_lookup = pd.DataFrame(stages_data["stages"])

#### Create json data


##### Standings


In [25]:
standings = []

for group in groups_lookup["name"]:
    request = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/html/2026_FIFA_World_Cup_{group.replace(' ', '_')}",
        headers={
            "User-Agent": f"World-Cup-2026-App/1.0 ({dotenv_values('../.env')['EMAIL']})"
        },
    )

    soup = BeautifulSoup(request.text, "html.parser")
    table = pd.read_html(StringIO(str(soup.find_all("table", {"class": "wikitable"}))))[
        1
    ]

    # Remove host tag from team name
    table["Teamvte"] = table["Teamvte"].str.replace(r"\s*\(H\)", "", regex=True)

    # Add teamId and short team name
    team_ids = []
    short_names = []
    flags = []
    for team in table["Teamvte"]:
        team_info = teams_lookup[
            (teams_lookup["fullName"] == team) | (teams_lookup["shortName"] == team)
        ].reset_index(drop=True)
        # General cases
        if not team_info.empty:
            team_ids.append(team_info.loc[0, "id"] if len(team_info) > 0 else None)
            short_names.append(
                team_info.loc[0, "shortName"] if len(team_info) > 0 else None
            )
            flags.append(team_info.loc[0, "flag"] if len(team_info) > 0 else None)
        # Special cases
        else:
            # Wikipedia name - FIFA-recognised name
            special_cases = {
                "Cape Verde": "Cabo Verde",
                "DR Congo": "Congo DR",
                "Turkey": "Türkiye",
            }
            team_info = teams_lookup[
                teams_lookup["fullName"] == special_cases.get(team, team)
            ].reset_index(drop=True)
            team_ids.append(team_info.loc[0, "id"] if len(team_info) > 0 else None)
            short_names.append(
                team_info.loc[0, "shortName"] if len(team_info) > 0 else None
            )
            flags.append(team_info.loc[0, "flag"] if len(team_info) > 0 else None)

    # Add teamId column to table
    table["id"] = pd.Series(team_ids)
    table["shortName"] = pd.Series(short_names)
    table["flag"] = pd.Series(flags)

    # Some pre-processing steps
    ## Rename columns
    table.rename(
        columns={
            "Teamvte": "teamName",
            "Pos": "position",
            "Pld": "played",
            "W": "wins",
            "D": "draws",
            "L": "losses",
            "GF": "goalsScored",
            "GA": "goalsConceded",
            "GD": "goalDiff",
            "Pts": "points",
        },
        inplace=True,
    )

    ## Drop unnecessary columns
    table.drop(columns=["Qualification"], inplace=True, errors="ignore")

    ## Move id column in front of teamName
    cols = table.columns.tolist()
    cols.insert(0, cols.pop(cols.index("id")))
    table = table[cols]

    # Convert to dict and add to standings
    standings.append(
        {
            "id": groups_lookup[groups_lookup["name"] == group]["id"].iloc[0],
            "name": group,
            "teams": table.to_dict(orient="records"),
        }
    )

In [26]:
# Write standings to JSON file
with open("../data/standings.json", "w", encoding="utf-8") as f:
    json.dump({"standings": standings}, f, indent=3, ensure_ascii=False)
    f.close()

##### Matches/Results


In [27]:
request = requests.get(
    f"https://en.wikipedia.org/api/rest_v1/page/html/2026_FIFA_World_Cup_Group_A",
    headers={
        "User-Agent": f"World-Cup-2026-App/1.0 ({dotenv_values('../.env')['EMAIL']})"
    },
)

soup = BeautifulSoup(request.text, "html.parser")

In [53]:
test_all_matches = soup.find_all("section")[3].find_all("section")
test_match = test_all_matches[0]

In [51]:
(uuid.uuid4().hex)[:16]

'3f7b1a593eb24a50'

In [104]:
matches = []

# For now, just defaulting to group stage
for group in groups_lookup["name"]:
    request = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/html/2026_FIFA_World_Cup_{group.replace(' ', '_')}",
        headers={
            "User-Agent": f"World-Cup-2026-App/1.0 ({dotenv_values('../.env')['EMAIL']})"
        },
    )

    soup = BeautifulSoup(request.text, "html.parser")
    all_matches = soup.find_all("section")[3].find_all("section")

    for match in all_matches:
        # Match info
        ## Fill in what's available
        match_info = {
            "id": (uuid.uuid4().hex)[:16],
            "description": match.find("h3").text.strip(),
            "localStartDate": (
                match.find("time")
                .find_all("span", {"class": "bday dtstart published updated itvstart"})[
                    0
                ]
                .text.strip()
            ),
            "localStartTime": "",
            "stage": {
                "id": stages_lookup[stages_lookup["name"] == "Group Stage"]["id"].iloc[
                    0
                ],
                "name": "Group Stage",
                "group": {
                    "id": groups_lookup[groups_lookup["name"] == group]["id"].iloc[0],
                    "name": group,
                },
            },
            "contestants": [
                {
                    "id": "",
                    "name": "",
                    "position": "home",
                },
                {
                    "id": "",
                    "name": "",
                    "position": "away",
                },
            ],
            "venue": match.find("span", {"itemprop": "name address"}).text.strip(),
        }

        ## Extract the time string and clean it
        time_str = (
            match.find("time")
            .find("div", {"class": "ftime"})
            .text.replace("\xa0", " ")
            .replace(".", "")
            .strip()
        )

        ## Remove timezone if found
        if "UTC" in time_str:
            time_str = time_str.rsplit(" ", 1)[0].strip()

        ## Convert to 24-hour
        time_24h = pd.to_datetime(time_str, format="%I:%M %p").strftime("%H:%M")
        match_info["localStartTime"] = time_24h

        ## Get the contestants
        home_team = match.find("h3").text.strip().split(" vs ")[0]
        away_team = match.find("h3").text.strip().split(" vs ")[1]

        match_info["contestants"][0]["name"] = home_team
        match_info["contestants"][1]["name"] = away_team

        ## Get contestant IDs
        ### General cases
        if (
            not teams_lookup[
                (teams_lookup["fullName"] == home_team)
                | (teams_lookup["shortName"] == home_team)
            ].empty
            and not teams_lookup[
                (teams_lookup["fullName"] == away_team)
                | (teams_lookup["shortName"] == away_team)
            ].empty
        ):
            home_team_info = teams_lookup[
                (teams_lookup["fullName"] == home_team)
                | (teams_lookup["shortName"] == home_team)
            ].reset_index(drop=True)
            away_team_info = teams_lookup[
                (teams_lookup["fullName"] == away_team)
                | (teams_lookup["shortName"] == away_team)
            ].reset_index(drop=True)
        ### Special cases
        else:
            # Wikipedia name - FIFA-recognised name
            special_cases = {
                "Cape Verde": "Cabo Verde",
                "DR Congo": "Congo DR",
                "Turkey": "Türkiye",
            }
            home_team_info = teams_lookup[
                teams_lookup["fullName"] == special_cases.get(home_team, home_team)
            ].reset_index(drop=True)
            away_team_info = teams_lookup[
                teams_lookup["fullName"] == special_cases.get(away_team, away_team)
            ].reset_index(drop=True)

        match_info["contestants"][0]["id"] = (
            home_team_info.loc[0, "id"] if len(home_team_info) > 0 else None
        )
        match_info["contestants"][1]["id"] = (
            away_team_info.loc[0, "id"] if len(away_team_info) > 0 else None
        )

        # -------------------------------------------------------------------------
        # Match data
        match_data = {
            "matchStatus": (
                "Fixture"
                if "Match" in match.find("th", {"class": "fscore"}).text.strip()
                else "Played"
            ),
            "matchLengthMin": "",
            "matchLengthSec": "",
            "period": [
                {
                    "id": 1,
                    "lengthMin": "",
                    "lengthSec": "",
                    "stoppageTime": "",  # In seconds
                },
                {
                    "id": 2,
                    "lengthMin": "",
                    "lengthSec": "",
                    "stoppageTime": "",  # In seconds
                },
            ],
            "scores": {
                "ht": {
                    "home": 0,
                    "away": 0,
                },
                "ft": {
                    "home": 0,
                    "away": 0,
                },
                "et": {
                    "home": 0,
                    "away": 0,
                },
                "total": {
                    "home": (
                        len(
                            match.find("td", {"class": "fhgoal"})
                            .text.strip()
                            .split("\n")
                        )
                        if match.find("td", {"class": "fhgoal"}).text != ""
                        else 0
                    ),
                    "away": (
                        len(
                            match.find("td", {"class": "fagoal"})
                            .text.strip()
                            .split("\n")
                        )
                        if match.find("td", {"class": "fagoal"}).text != ""
                        else 0
                    ),
                },
            },
        }

        # -------------------------------------------------------------------------
        # Combine match info and data, and add to matches list
        matches.append({"matchInfo": match_info, "matchData": match_data})

In [106]:
# Write matches to JSON file
with open("../data/matches.json", "w", encoding="utf-8") as f:
    json.dump({"matches": matches}, f, indent=3, ensure_ascii=False)
    f.close()

##### Squads
